# Take-home 01: the model format

Due September 24th, 9:30am on Canvas. This assignment is graded, but you may work with your lab partner if you include your name on the submission.

Every model you build this semester will have the same three parts: a `Dataset`, a `DataLoader`, and a training loop written as an `nn.Module`. Below, each part gets built once, on four points where you already know the right answer. Then it's yours: five short exercises, each a different rep of the same three parts.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import seaborn as sns

penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)

## Dataset

A `Dataset` bundles your input and target tensors into one object. It answers two questions: how many examples do you have, and what is example number `i`. Keeping `x` and `y` bundled like this matters once batching and shuffling enter: PyTorch keeps every input paired with its own target automatically, so you never have to line them up by hand.

In [ ]:
x_demo = torch.tensor([1.0, 2.0, 3.0, 4.0])
y_demo = torch.tensor([3.0, 5.0, 7.0, 9.0])

ds_demo = TensorDataset(x_demo, y_demo)
len(ds_demo)       # 4 examples
ds_demo[0]         # (1.0, 3.0), one input and its target, paired

## DataLoader

A `DataLoader` wraps a `Dataset` and hands you batches when you loop over it, instead of handing you one example at a time. `batch_size=len(ds_demo)` with `shuffle=False` asks for one batch containing everything, in the original order, which behaves exactly like handing your hand-rolled loop the whole dataset at once, the same way weeks 2 and 3 both did it.

In [ ]:
dl_demo = DataLoader(ds_demo, batch_size=len(ds_demo), shuffle=False)
for xb, yb in dl_demo:
    print(xb, yb)

## The training loop, as `nn.Module`

`nn.Module` is a base class that holds a model's parameters and its forward pass together, as one object, instead of loose tensors scattered around your notebook. `nn.Parameter` marks a tensor as one the model should track; `model.parameters()` then hands back exactly those tensors, so an optimizer knows what to update without you naming each one. `forward` is the computation itself, called automatically whenever you write `model(x)`.

In [ ]:
class LinearModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.tensor(0.0))
        self.b = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        return self.w * x + self.b


model = LinearModel()
opt = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(300):              # 300 passes over the data
    for xb, yb in dl_demo:            # one batch per pass (here, the whole dataset)
        pred = model(xb)              # forward pass
        loss = ((pred - yb) ** 2).mean()   # score it
        opt.zero_grad()               # clear old gradients
        loss.backward()               # compute new ones
        opt.step()                    # update w and b

print(f"w={model.w.item():.3f} b={model.b.item():.3f} loss={loss.item():.5f}")
# lands near w=2, b=1: y_demo was 2x+1 exactly

Same five moves week 2 wrote by hand: predict, score, clear, backward, step. They haven't changed. What changed is that `w` and `b` now live inside `model`, and the update now lives inside `opt`, instead of being four loose lines you had to get right every time.

`forward` above has no activation: the prediction is just the weighted sum, `w * x + b`. That's called the identity, the simplest possible choice, and it's what every linear model you've fit so far has been using without a name for it. Tuesday's lecture goes deep on what else can go in that slot.

## Exercises

Five reps, real data throughout. Fill in each `...`.

### Exercise 1: build a `Dataset` and `DataLoader`, nothing else

You already know how to plot and inspect penguins columns. This exercise is just the `Dataset`/`DataLoader` mechanics on your own, with no model attached yet.

In [ ]:
bill_length = torch.tensor(penguins["bill_length_mm"].to_numpy(), dtype=torch.float32)
bill_depth = torch.tensor(penguins["bill_depth_mm"].to_numpy(), dtype=torch.float32)

In [ ]:
# YOUR TURN:
#   1. wrap bill_length and bill_depth in a TensorDataset, call it ds0
#   2. wrap ds0 in a DataLoader, call it dl0: batch_size=50, shuffle=True
# Then run the loop below as it is and look at what it prints.
ds0 = ...
dl0 = ...

for xb, yb in dl0:
    print(xb.shape, yb.shape)

333 penguins at `batch_size=50` should print 7 lines: six batches shaped `(50,)`, and one last batch shaped `(33,)`, the leftover. If you see a different number of lines, or every batch the same size, `shuffle` or `batch_size` isn't set the way the comment asks.

### Exercise 2: wrap week 2's fit

Week 2 fit `flipper_length_mm` against `body_mass_g` (kg) by hand: `w=0`, `b=0`, `lr=1e-5`, 2000 steps, full-batch. This is the first time you rebuild an entire fit you already trust, end to end, in the organized form.

In [ ]:
flipper = torch.tensor(penguins["flipper_length_mm"].to_numpy(), dtype=torch.float32)
mass = torch.tensor(penguins["body_mass_g"].to_numpy(), dtype=torch.float32) / 1000

In [ ]:
# YOUR TURN:
#   1. wrap flipper and mass in a TensorDataset, call it ds1
#   2. wrap ds1 in a DataLoader, call it dl1: batch_size=len(ds1), shuffle=False
ds1 = ...
dl1 = ...

In [ ]:
# YOUR TURN: build a LinearModel (the class above, reused as-is) and train it on dl1:
#   optimizer: SGD, lr=1e-5
#   loop 2000 times, same five moves as the demo above
model1 = ...


print(f"loss={loss1.item():.5f} w={model1.w.item():.5f} b={model1.b.item():.5f}")

Reference, both hand-rolled and organized: loss lands near **0.32014**, at **w=0.02108, b=-0.00103**. That loss is mediocre, and it's supposed to be: this is the exact fit from week 2's lab, where flipper length's raw scale (around 200) made the gradients unmanageable at a learning rate small enough to survive. Rebuilding it here isn't fixing that problem; it's checking that your organized version computes the same thing your hand-rolled version did, mediocre fit and all.

### Exercise 3: a model that takes two features

Everything so far has been one number in, one number out. Predict `body_mass_g` from **two** features at once: `bill_length_mm` and `bill_depth_mm`, the tensors you already built in Exercise 1.

In [ ]:
X = torch.stack([bill_length, bill_depth], dim=1)   # shape (333, 2): one row per penguin, two columns
mass2 = torch.tensor(penguins["body_mass_g"].to_numpy(), dtype=torch.float32) / 1000

`LinearModel` needs exactly two changes to take two features instead of one: `self.w` becomes a length-2 vector, one weight per feature, and the weighted sum becomes a matrix multiply, `x @ self.w`, instead of `self.w * x`.

In [ ]:
# YOUR TURN: copy LinearModel below, call it LinearModel2.
#   self.w = nn.Parameter(torch.zeros(2))    -- one weight per feature, not one number
#   forward returns x @ self.w + self.b      -- matrix multiply, not scalar multiply
class LinearModel2(nn.Module):
    ...

In [ ]:
# YOUR TURN: Dataset + DataLoader on X, mass2 (full batch, shuffle=False).
# Train LinearModel2: SGD, lr=1e-4, 2000 steps.
model3 = ...


print(f"loss={loss3.item():.5f} w={model3.w.detach().numpy()} b={model3.b.item():.4f}")

Reference: loss lands near **0.407**, at **w near [0.110, -0.038]**. Same story as Exercise 2: these two features are also raw-scale (bill length in the 30s-50s, bill depth in the teens), so the fit plateaus rather than converging cleanly. That's not the point here either; the point is that going from one feature to two cost exactly two small changes, not a rewrite.

### Exercise 4: one line makes it logistic regression

Two penguin species, Gentoo and Adelie, and one feature: body mass in kg (a convenient scale already, no rescaling needed).

In [ ]:
sub = penguins[penguins["species"].isin(["Gentoo", "Adelie"])].reset_index(drop=True)
x2 = torch.tensor(sub["body_mass_g"].to_numpy(), dtype=torch.float32) / 1000
t2 = torch.tensor((sub["species"] == "Gentoo").to_numpy(), dtype=torch.float32)

`torch.sigmoid` squashes any number into `(0, 1)`, so its output reads as a probability. Wrapping the weighted sum in it, instead of returning the raw sum, is the entire difference between a linear model and a logistic one.

In [ ]:
# YOUR TURN: copy LinearModel below, call it LogisticModel.
#   __init__ is identical.
#   forward returns torch.sigmoid(self.w * x + self.b) instead of the raw sum.
class LogisticModel(nn.Module):
    ...

In [ ]:
# YOUR TURN: build ds2/dl2 from x2, t2 (same shape as Exercise 2: full batch, shuffle=False).
#   Train LogisticModel: SGD, lr=0.5, 2000 steps. Loss is still squared error, targets still 0/1,
#   same as week 3.
model2 = ...


pred2 = model2(x2)
acc2 = ((pred2 > 0.5).float() == t2).float().mean()
print(f"loss={loss2.item():.5f} acc={acc2.item():.4f} w={model2.w.item():.4f} b={model2.b.item():.4f}")

Reference, both versions: loss near **0.07395**, accuracy near **0.9094**, at **w=2.3406, b=-10.3193**. Unlike Exercises 2 and 3, this fit is genuinely good, so a close match here tells you two things at once: the reorganization is faithful, and the model works.

### Exercise 5: give the `DataLoader` something to do

Same `LogisticModel`, same data as Exercise 4. This time, `batch_size=32` and `shuffle=True` instead of one full batch, for 200 epochs (one epoch is one full pass over all the shuffled batches) instead of 2000 full-batch steps.

In [ ]:
# YOUR TURN: build a new DataLoader over ds2: batch_size=32, shuffle=True.
#   New LogisticModel, SGD, lr=0.5. Loop 200 epochs; inside each epoch, loop over
#   every batch the loader hands you (a nested loop, same shape as the demo above,
#   just with more than one batch per epoch now).
model5 = ...


pred5 = model5(x2)
acc5 = ((pred5 > 0.5).float() == t2).float().mean()
print(f"loss={loss5.item():.5f} acc={acc5.item():.4f}")

Reference range: loss around 0.09-0.10, accuracy around 0.85-0.87, a little worse than Exercise 4's. Yours will land somewhere in that range and won't match a neighbor's exactly, and that's expected: with smaller shuffled batches, each update sees only a slice of the data rather than all of it, so the path to a fit looks a little different every run even though the destination is similar. That's the shuffle, not a bug.

## Check zone, graded

Make a cell for each answer, and put the answer in that cell. Feel free to use code to compute your answer, but ensure you explain your reasoning. The grader will only look at the text you put in the answer cell.

1. **Multiple choice.** Exercise 1's `DataLoader` produced 7 batches from 333 penguins at `batch_size=50`. Why 7, and not 6? (a) `DataLoader` always rounds up, (b) the last batch holds whatever's left over, 33 examples, (c) `shuffle=True` adds an extra batch, (d) `TensorDataset` pads the data to a multiple of 50. **Answer (b).**

2. **Multiple choice.** Exercise 2's hand-rolled and organized versions land on close to the same numbers, even though the fit itself was mediocre. What does that match confirm? (a) the fit is correct, (b) the reorganization computes the same thing the hand-rolled loop did, (c) `nn.Module` fixed the learning-rate problem, (d) `DataLoader` improved convergence. **Answer (b).**

3. **Short.** Exercise 2's fit doesn't reach a good loss in 2000 steps. Does that mean your organized version is wrong? Why or why not?

4. **Short.** Exercise 3 took two features instead of one. Name the two things that had to change about `LinearModel` to make that work.

5. **Multiple choice.** Exercise 4 turned `LinearModel` into `LogisticModel`. What changed in the code? (a) the loss function, (b) the optimizer, (c) one line in `forward`, wrapping the output in `sigmoid`, (d) the `DataLoader`. **Answer (c).**

6. **Short.** Exercise 5's numbers differ from Exercise 4's, even though the model and the underlying data are the same. What changed, and why would that change the result?

7. **Short.** Where does an activation function sit in this week's `nn.Module` models, and what does `LinearModel` use there even though it doesn't visibly do anything?

Q1:

Q2:

Q3:

Q4:

Q5:

Q6:

Q7: